# Análise de operações financeiras - Nível 1
## Parte A - Análise determinística com Pandas

In [1]:
import json
from pathlib import Path
import pandas as pd

In [ ]:
caminho_dados = Path("../dados/dados_nivel_1.json")
Path.cwd()
caminho_dados.exists()

True

In [20]:

with open(caminho_dados, "r", encoding="utf-8") as arquivo:
    dados = json.load(arquivo)

taxa_cambio_usd_brl = dados["taxa_cambio_usd_brl"]

df = pd.DataFrame(dados["operacoes"])
df


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [21]:
df.isna().sum()


id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

In [22]:
df["id"].duplicated().sum()

np.int64(1)

In [23]:
df["moeda"].value_counts(dropna=False)
df["canal"].value_counts(dropna=False)
df["tipo"].value_counts(dropna=False)

tipo
transferencia_enviada     11
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64

In [ ]:
df["cliente_id"].nunique()

# 20 operacoes, e 6 clientes

6

In [29]:
df[df["id"].duplicated(keep=False)].sort_values("id")

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [40]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["data"].dtype

dtype('<M8[us]')

In [47]:
df = df.drop_duplicates().copy()

df["data"] = pd.to_datetime(df["data"], errors="coerce")

df["data_ausente"] = df["data"].isna()

In [48]:
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False


In [49]:
print("Quantidade de registros:", len(df))
print("Duplicidades integrais:", df.duplicated().sum())
print("Datas ausentes:", df["data"].isna().sum())
print("Clientes únicos:", df["cliente_id"].nunique())

Quantidade de registros: 19
Duplicidades integrais: 0
Datas ausentes: 1
Clientes únicos: 6


In [50]:
colunas_categoricas = ["canal", "tipo"]

for coluna in colunas_categoricas:
    df[coluna] = df[coluna].str.strip().str.lower()

In [51]:
for coluna in colunas_categoricas:
    print(f"\nValores de {coluna}:")
    print(df[coluna].value_counts(dropna=False))


Valores de canal:
canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64

Valores de tipo:
tipo
transferencia_enviada     10
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64


In [52]:
df["moeda"] = df["moeda"].str.strip().str.upper()

In [53]:
df["moeda"].value_counts(dropna=False)

moeda
BRL    18
USD     1
Name: count, dtype: int64

In [54]:
df["valor"].describe()

count       19.000000
mean     11194.736842
std       8085.272873
min       1400.000000
25%       4050.000000
50%       8800.000000
75%      17250.000000
max      27000.000000
Name: valor, dtype: float64

In [55]:
df["valor_brl"] = df["valor"]

In [ ]:
df["valor_brl"] = df["valor"].astype(float)

# Identificar operações em USD
mascara_usd = df["moeda"] == "USD"

# Converter USD para BRL
df.loc[mascara_usd, "valor_brl"] = (
    df.loc[mascara_usd, "valor"].astype(float)
    * float(taxa_cambio_usd_brl)
)

In [61]:
df
# versão final do dataframe após as normalizações

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,18100.0
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,False,17300.0
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,False,18800.0
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False,3300.0
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False,25900.0
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False,27000.0
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,17200.0
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,15200.0
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False,16100.0
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False,3800.0


In [62]:
print("Quantidade de registros:", len(df))
print("Clientes únicos:", df["cliente_id"].nunique())
print("Duplicidades integrais:", df.duplicated().sum())
print("Datas ausentes:", df["data"].isna().sum())
print("Valores BRL ausentes:", df["valor_brl"].isna().sum())
print("Moedas encontradas:", df["moeda"].unique())

Quantidade de registros: 19
Clientes únicos: 6
Duplicidades integrais: 0
Datas ausentes: 1
Valores BRL ausentes: 0
Moedas encontradas: <StringArray>
['BRL', 'USD']
Length: 2, dtype: str
